# nteract 2.0.0

We just shipped nteract 2.0.0 — a ground-up rebuild of the notebook app. New runtime, new architecture, new everything. Here's what we built and why.

## What is nteract?

nteract is a desktop notebook app for macOS (, Windows, and Linux. You double-click a `.ipynb` file, it opens, a kernel starts, and you're running code. No browser, no server, no `jupyter notebook` command.

We originally built nteract on Electron back in 2016. This is a complete rewrite in Rust and TypeScript on Tauri. The app is native, the kernels are managed by a background daemon, and some hidden magic.

## The big ideas

### Zero-config environments

Open a notebook, get a kernel. nteract figures out the environment automatically:

1. **Inline dependencies** — deps stored right in the notebook metadata. The notebook is fully portable. Hand it to someone, they open it, it installs what it needs.
2. **Project files** — `pyproject.toml`, `pixi.toml`, `environment.yml`. nteract walks up from the notebook directory and uses the closest match.
3. **Prewarmed pools** — no project file? The daemon keeps warm Python environments ready to go. First cell runs in under a second.

Both **UV** and **Conda** backends are supported. UV for fast pip-style installs, Conda for when you need system-level deps (CUDA, etc.). Deno notebooks work too — TypeScript and JavaScript with automatic Deno bootstrapping.

A trust system (HMAC-SHA256, per-machine key) prevents shared notebooks from silently installing packages. You review deps before they run.

### AI agents as first-class peers

This is the one I'm most excited about. nteract ships with an MCP server. Claude, ChatGPT, Gemini, Cursor, Zed, OpenCode — any MCP-capable AI can connect and work in the same live notebook as you.

```
pip install nteract
```

The agent gets 22+ tools: create cells, write code, execute, read outputs, manage dependencies, restart kernels. The human sees every change in real time. The agent sees every change the human makes. Same CRDT, same document, same truth.

```
AI agent ← MCP → nteract server → runtimed daemon ← nteract Desktop
                                    (real-time CRDT sync)
```

Multiple agents and multiple humans can all be in the same notebook simultaneously. Presence shows who's where — colored cursors, cell focus indicators, text attribution highlights with fade-out.

### Local-first notebooks

Every notebook is an [Automerge](https://automerge.org/) CRDT document. The frontend runs its own Automerge peer in WebAssembly. The daemon runs another. Cell edits happen locally in WASM — no round trip, no lag. The two peers sync over a binary pipe.

This means:
- **Instant editing.** Keystrokes don't wait for a server.
- **Multi-window.** Open the same notebook in two windows. They stay in sync.
- **Crash recovery.** The daemon persists the Automerge doc. If the app crashes, your work is there when you reopen.
- **Autosave.** The daemon writes `.ipynb` on a debounce. No manual save needed (though Cmd+S still works).

### The daemon: `runtimed`

A background service (`runtimed`) runs on your machine and manages everything:

- **Kernel lifecycle** — launch, interrupt, restart, shutdown. Process groups are tracked and reaped.
- **Output storage** — a content-addressed blob store for images, plots, HTML. The CRDT carries only SHA-256 hashes. A local HTTP server serves blobs to the frontend.
- **Environment management** — prewarmed pools, inline dep caching, project file detection.
- **Settings sync** — theme, default runtime, keepalive — synced via a separate Automerge doc across all windows.
- **Notebook rooms** — each open notebook is a "room" with autosave, eviction on disconnect, and crash recovery via persisted Automerge docs.

The daemon runs as a system service on macOS (`io.nteract.runtimed`). It starts automatically and stays out of your way.

## What you can do with it

### For humans

- Open `.ipynb` files natively — double-click from Finder/Explorer
- Inline dependency management — add packages without leaving the notebook
- Cell drag-and-drop, global find
- ipywidgets support
- Terminal emulation for stream output (try `!vim`, then interrupt it)
- Dark mode, synced settings, auto-update
- Cell visibility toggles

### For AI agents

Tell your agent of choice to add the mcp server that launches with `uvx nteract`.

Then your AI can:
- Create and manage notebooks
- Write and execute code
- Read outputs (text, images, errors)
- Install dependencies
- Pattern-match edit cells (`replace_match`, `replace_regex`)
- Restart kernels, manage environment

## Install

**Desktop app:**

[nteract.io](https://www.nteract.io)

macOS DMG is signed and notarized. Linux has AppImage and .deb. Windows has an installer (not code-signed yet — expect a SmartScreen prompt).

**MCP server (for AI agents):**
```
pip install nteract
```

**Python bindings:**

Use these to make your own opinionated 

```
pip install runtimed
```

## What's next

Widget state in the CRDT. More environment backends. Code signing on Windows. And whatever you build with it.

[github.com/nteract/desktop](https://github.com/nteract/desktop)


In [1]:
!pwd

/Users/kylekelley/code/src/github.com/nteract/desktop


In [1]:
import time

time.sleep(100)